# IMPORT LIBRARIES

In [ ]:
import os
from io import BytesIO
from PIL import Image
import librosa
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import cv2
import warnings
matplotlib.use('Agg')

data_path_Train = pd.read_csv("CSVs\\cremad_augmented.csv")
data_path_Test = pd.read_csv("CSVs\\cremad_test.csv")

# FEATURE EXTRACTION

In [ ]:
def extract_features(ef_data: np.ndarray, ef_sample_rate: int) -> np.ndarray:

    ef_features = np.array([])
    n_fft = min(2048, len(ef_data))
    n_mels = min(128, n_fft // 2)

    # ZCR
    zcr = np.mean(librosa.feature.zero_crossing_rate(ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, zcr))

    # Chroma STFT
    stft = librosa.stft(ef_data, n_fft=n_fft)
    chroma_stft = np.mean(
        librosa.feature.chroma_stft(S=np.abs(stft), sr=ef_sample_rate, n_fft=n_fft).T, axis=0
    )
    ef_features = np.hstack((ef_features, chroma_stft))

    # MFCC
    mfcc = np.mean(librosa.feature.mfcc(y=ef_data, sr=ef_sample_rate, n_fft=n_fft).T, axis=0)
    ef_features = np.hstack((ef_features, mfcc))

    # Root Mean Square Value
    rmse = np.mean(librosa.feature.rms(y=ef_data).T, axis=0)
    ef_features = np.hstack((ef_features, rmse))

    # Mel Spectrogram
    mel = np.mean(
        librosa.feature.melspectrogram(y=ef_data, sr=ef_sample_rate, n_fft=n_fft, n_mels=n_mels).T, axis=0
    )
    ef_features = np.hstack((ef_features, mel))

    return ef_features


def get_features(gf_path: str) -> np.ndarray:
    data, sample_rate = librosa.load(gf_path, sr=None)

    sample_rate = int(sample_rate)
    features = extract_features(data, sample_rate)

    return features


MIN_SAMPLES = 2048

def prepare_audios(pa_df: pd.DataFrame, pa_name: str):
    data = []
    labels = []
    skipped = 0
    total = len(pa_df)

    for _, row in pa_df.iterrows():
        try:
            audio, sr = librosa.load(row["path"], sr=None)

            if len(audio) < MIN_SAMPLES:
                skipped += 1
                print(f"Skipped (too short): {row['path']} ({len(audio)} samples)")
                continue

            features = extract_features(audio, int(sr))
            data.append(features)
            labels.append(row["emotion"])
            print(f"{pa_name} - Done! Saved {len(data)}/{total}", end="\r", flush=True)

        except Exception as e:
            print(f"Error processing {row['path']}: {e}")

    print(f"\n{pa_name} - Hoàn tất: {len(data)} files, bỏ qua: {skipped} files")

    data = np.array(data)
    labels = np.array(labels)

    os.makedirs("features", exist_ok=True)
    np.save(f"features\\{pa_name}_data.npy", data)
    np.save(f"features\\{pa_name}_labels.npy", labels)


prepare_audios(data_path_Train, "train")
prepare_audios(data_path_Test, "test")

# MEL SPECTROGRAM

In [4]:
# CONFIG
location = "features\\images\\"
csv_dir = "CSVs\\"

MIN_SAMPLES = 2048
TARGET_DURATION = 3.0  # seconds
IMG_SIZE = 224

# Mel params (good for speech)
N_MELS = 128
N_FFT = 1024
HOP_LENGTH = 256


# FIX AUDIO LENGTH
def fix_length(audio: np.ndarray, sr: int) -> np.ndarray:
    target_len = int(sr * TARGET_DURATION)

    if len(audio) > target_len:
        audio = audio[:target_len]

    elif len(audio) < target_len:
        pad = target_len - len(audio)
        audio = np.pad(audio, (0, pad))

    return audio


# AUDIO -> MEL IMAGE
def graph_spectrogram(gs_audio, sr: int | float | None = None) -> np.ndarray:

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")

        # If input is path
        if isinstance(gs_audio, str):
            audio, loaded_sr = librosa.load(gs_audio, sr=None)
            sr = int(loaded_sr)

        else:
            audio = gs_audio
            if sr is None:
                raise ValueError("sr required when input is waveform")
            sr = int(sr)

        # fix duration
        audio = fix_length(audio, sr)

        # mel spectrogram
        mel = librosa.feature.melspectrogram(
            y=audio, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH
        )

        mel_db = librosa.power_to_db(mel, ref=np.max)

        # plot no border
        fig, ax = plt.subplots(figsize=(2.24, 2.24))
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        ax.axis("off")

        librosa.display.specshow(
            mel_db, sr=sr, hop_length=HOP_LENGTH, cmap="inferno", ax=ax
        )

        buf = BytesIO()
        fig.savefig(buf, format="png", dpi=100)
        buf.seek(0)

        img = np.array(Image.open(buf))[:, :, :3]

        plt.close(fig)
        buf.close()

        return img


# CREATE IMAGE DATASET
def prepare_images(pi_df: pd.DataFrame, pi_name: str):
    save_dir = os.path.join(location, pi_name)
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(csv_dir, exist_ok=True)

    file_data = []
    skipped = 0
    counter = 1
    total = len(pi_df)

    for _, row in pi_df.iterrows():
        try:
            raw, sr = librosa.load(row["path"], sr=None)

            if len(raw) < MIN_SAMPLES:
                skipped += 1
                continue

            img = graph_spectrogram(raw, sr)

            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            img_path = os.path.join(save_dir, f"{counter}.png")
            cv2.imwrite(img_path, img)

            file_data.append([row["speaker"], row["emotion"], img_path])

            print(f"\r{pi_name}: {counter}/{total}", end="", flush=True)

            counter += 1

        except Exception as e:
            print(f"\nError: {row['path']} -> {e}")

    print(f"\n{pi_name} done | saved={counter - 1} | skipped={skipped}")

    result_df = pd.DataFrame(file_data, columns=["speaker", "emotion", "path"])

    result_df.to_csv(os.path.join(csv_dir, f"{pi_name}_images.csv"), index=False)


# RUN
prepare_images(data_path_Train, "train")
prepare_images(data_path_Test, "test")

train: 11780/11780
train done | saved=11780 | skipped=0
test: 1552/1552
test done | saved=1552 | skipped=0
